# Checks & Debugs regarding refactoring of code
Felix Zaussinger | XX.YY.ZZZZ

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths(fn_config_path="paths_config.yml")

In [14]:
from src.data.framework import Esco, Onet
esco = Esco()
onet = Onet()

ESCO Class

In [18]:
esco.occupations
esco.skills

,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage staff of music\ncoordinate duties of mu...,NaN,released,2016-12-20T17:43:43Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,oversee prison procedures\nmanage correctional...,NaN,released,2016-12-20T20:17:49Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Supervise the operations of a correctional fac...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,apply non-oppressive practices\napply an anti-...,NaN,released,2016-12-20T19:18:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c..."
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,monitoring of compliance with railway vehicles...,NaN,released,2016-12-20T20:02:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ..."
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,establish available services\ndetermine rehabi...,NaN,released,2016-12-20T20:15:17Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Identify the different services available for ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13886,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/ffef5eb3-a15e...,skill/competence,sector-specific,remediate healthcare user's occupational perfo...,restore healthcare user's occupational perform...,NaN,released,2016-12-20T19:25:53Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,"Remediate or restore the cognitive, sensorimot..."
13887,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0b074-5a76...,skill/competence,sector-specific,install transport equipment lighting,install transport equipment illumination\nfix ...,NaN,released,2016-12-20T20:03:21Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Install lighting elements in transport equipme...
13888,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0e2cd-d0bd...,knowledge,sector-specific,natural language processing,NLP,NaN,released,2016-08-04T15:19:37Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,The technologies which enable ICT devices to u...
13889,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff5bc45-b506...,skill/competence,cross-sector,coordinate construction activities,reviewing construction progress\nconstruction ...,NaN,released,2016-12-20T18:22:35Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Coordinate the activities of several construct...


Onet Class

In [3]:
onet.green_occupations_gtp
onet.green_occupations_vona2018
onet.green_tasks_gtp
onet.brown_occupations_vona2018
onet.green_occupations_narrow_jrc

,isco08_jrc,occ_eng_jrc,n_green_tasks_jrc,n_tasks_jrc,greenness_jrc
0,1.1.2.4.3,General managers and equivalent in health care,1,11.0,0.090909
1,1.1.2.6.3,Managers and equivalent in health care,1,13.0,0.076923
2,1.2.2.3.0,Directors and general managers of construction...,1,13.0,0.076923
3,1.2.3.2.0,"Directors and managers of the organisation, hu...",1,15.0,0.066667
4,1.2.3.7.0,Directors and managers of the research and dev...,1,12.0,0.083333
5,1.2.3.9.0,Other departmental directors and managers,1,10.0,0.100000
6,1.3.1.2.0,Entrepreneurs and managers of small companies ...,1,14.0,0.071429
7,1.3.1.3.0,Entrepreneurs and managers of small constructi...,1,14.0,0.071429
8,1.3.1.8.0,Entrepreneurs and managers of small companies ...,1,13.0,0.076923
9,2.1.1.2.1,Chemists and related professions,1,14.0,0.071429


Calculating occ-skills matrix with more flexibility

In [39]:
osm_weighted = esco.read_occ_skills_matrix(return_version="weighted")
np.unique(osm_weighted)

array([0. , 0.5, 1. ])

In [40]:
osm_unweighted = esco.read_occ_skills_matrix(return_version="unweighted", assign_labels=True)
np.unique(osm_unweighted)

array([0, 1], dtype=int64)

In [42]:
osm_encoded = esco.read_occ_skills_matrix(return_version="raw", assign_labels=True)
np.unique(osm_encoded)

array([0, 1, 2], dtype=int64)

In [22]:
osm_weighted = esco.label_osm(osm_weighted)
osm_weighted

,manage musical staff,supervise correctional procedures,apply anti-oppressive practices,control compliance of railway vehicles regulations,identify available services,perform toxicological studies,ensure coquille uniformity,Haskell,show initiative,train staff to reduce food waste,apply diplomatic principles,lead police investigations,handle fish harvesting waste,develop energy saving concepts,perform street interventions in social work,work with soloists,sport and exercise medicine,conduct research on flora,install heat pump,design biomass installations,handle equipment while suspended,teach housekeeping skills,check train engines,influence public policies,enterprise risk management,manufacture ingredients,maintain aquaculture ponds,apply credit risk policy,handle customer requests related to cargo,draft scientific or academic papers and technical documentation,Incremental development,use of special equipment for daily activities,sawing techniques,produce guitar components,operate agricultural machinery,control pyrotechnics stock,guarantee customer satisfaction,manufacture wearing apparel products,cure tobacco leaves,develop a rehabilitation programme,maintain inventory of cleaning supplies,cold vulcanisation,supervise housekeeping operations,act as contact person during equipment incident,manage time in landscaping,advise on customs regulations,manage university department,develop terminology databases,assess nutritional characteristics of food,repair electric bicycles,maintain sorting equipment,explain features in accommodation venue,types of barley,KDevelop,purchase vehicle parts,inspect offshore constructions,transport patient to medical facility,adjust envelope cutting settings,prepare oils,purchase supplies,...,establish gaming policies,media and information literacy,evaluate clinical outcomes of dental hygiene interventions,gem cutting forms,cut pig's teeth,provide therapy of the visual system,administer materials to tea bag machines,advise on bank account,manage dental emergencies,assume highest level of responsibility in inland water transportation,manage time in food processing operations,student financial aid programmes,interpret pedigree charts,perform software recovery testing,astronomy,use instruments for food measurement,tend ball mill,understand spoken Norwegian,cut filament,procure time sheet approval,communicate with a non-scientific audience,prepare soda-ash,microchip scanners,rope lashing,create social alliances,assess your competencies in leading community arts,write Occitan,arrange customs inspection,perform nutrition analysis,analyse work-related written reports,surveillance methods,remove old caulking,keep company,clean patients' ear canals,anodising process,preserve samples,treat vehicle fabrics,collect domestic waste,perform energy simulations,match vessels to shipping routes,types of drill bits,cable-propelled transit,negotiate rights of use,Capture One,precious metal processing,control train movement,dependency on drugs,organise vehicle parts storage,set up reinforcing steel,model sensor,Scala,operate forestry equipment,test soil load bearing capacity,buy new library items,design clocks,remediate healthcare user's occupational performance,install transport equipment lighting,natural language processing,coordinate construction activities,position guardrails and toeboards
technical director,0.0,0,0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
metal drawing machine operator,0.0,0,0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0

In [27]:
osm_weighted[osm_weighted == 1].sum(axis=1).describe()

count    3008.000000
mean       21.640957
std        12.145173
min         0.000000
25%        14.000000
50%        19.000000
75%        26.000000
max       133.000000
dtype: float64

In [28]:
osm_weighted[osm_weighted == 0.5].sum(axis=1).describe()

count    3008.000000
mean        9.766622
std        10.787132
min         0.000000
25%         4.500000
50%         8.500000
75%        11.500000
max       159.500000
dtype: float64

In [30]:
osm_unweighted[osm_unweighted == 1].sum(axis=1).describe()

count    3008.000000
mean       41.174202
std        24.990325
min         4.000000
25%        27.000000
50%        36.000000
75%        48.000000
max       340.000000
dtype: float64

Combining skills metadata

In [47]:
smd = esco.combine_skills_metadata(variable_selection=None)
smd

,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description,broaderConceptPT,machineScore_green,skill_green_esco,scope_eth,impact_eth,skill_green_eth,comment_eth,broaderConceptUri,alternative_class_eth,scope_eth_brown,skill_brown_esco,impact_eth_brown,comment_eth_brown,machineScore_brown,skill_brown_eth,alternative_class_eth_brown,skill_neutral_esco,skill_classification_esco,coreness
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage staff of music\ncoordinate duties of mu...,NaN,released,2016-12-20T17:43:43Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.002187
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,oversee prison procedures\nmanage correctional...,NaN,released,2016-12-20T20:17:49Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Supervise the operations of a correctional fac...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.000000
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,apply non-oppressive practices\napply an anti-...,NaN,released,2016-12-20T19:18:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c...",NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.000748
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,monitoring of compliance with railway vehicles...,NaN,released,2016-12-20T20:02:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ...",NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.049473
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,establish available services\ndetermine rehabi...,NaN,released,2016-12-20T20:15:17Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Identify the different services available for ...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.001577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13886,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/ffef5eb3-a15e...,skill/competence,sector-specific,remediate healthcare user's occupational perfo...,restore healthcare user's occupational perform...,NaN,released,2016-12-20T19:25:53Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,"Remediate or restore the cognitive, sensorimot...",NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.000103
13887,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0b074-5a76...,skill/competence,sector-specific,install transport equipment lighting,install transport equipment illumination\nfix ...,NaN,released,2016-12-20T20:03:21Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Install lighting elements in transport equipme...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.013943
13888,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0e2cd-d0bd...,knowledge,sector-specific,natural language processing,NLP,NaN,released,2016-08-04T15:19:37Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,The technologies which enable ICT devices to u...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,True,neutral,0.025162
13889,Knowledg

Calculating GBN shares at occupation level

In [48]:
esco.calc_gbn_shares_skill_based(skills_metadata=smd)

,conceptUri,n_total_specific_skills,n_green_specific_skills_esco,share_green_esco,n_brown_specific_skills_esco,share_brown_esco,n_neutral_specific_skills_esco,share_neutral_esco,gbn_classification_esco
0,http://data.europa.eu/esco/occupation/00030d09...,8,0,0.000000,0,0.000000,8,1.000000,neutral
1,http://data.europa.eu/esco/occupation/000e93a3...,39,0,0.000000,5,0.128205,34,0.871795,neutral
2,http://data.europa.eu/esco/occupation/0019b951...,43,0,0.000000,0,0.000000,43,1.000000,neutral
3,http://data.europa.eu/esco/occupation/0022f466...,39,2,0.051282,0,0.000000,37,0.948718,neutral
4,http://data.europa.eu/esco/occupation/002da35b...,26,0,0.000000,0,0.000000,26,1.000000,neutral
...,...,...,...,...,...,...,...,...,...
3003,http://data.europa.eu/esco/occupation/ff656b3a...,58,0,0.000000,0,0.000000,58,1.000000,neutral
3004,http://data.europa.eu/esco/occupation/ff8d4065...,26,14,0.538462,0,0.000000,12,0.461538,green
3005,http://data.europa.eu/esco/occupation/ffa4dd5d...,29,0,0.000000,1,0.034483,28,0.965517,neutral
3006,http://data.europa.eu/esco/occupation/ffade2f4...,31,0,0.000000,0,0.000000,31,1.000000,neutral
